# 08 Commune segmentation

Group communes by their infrastructure profile, then check whether the lag rate differs across the segments. The point isn't to predict ,  that's notebook 07's job ,  it's to ask whether the structural features (PM density, building mix, scale) carve the metropolis into distinct types of place, and whether one of those types is systematically left behind.

I use sklearn's `KMeans` rather than UMAP + HDBSCAN. With ~80 communes the sample is too small for density-based clustering to behave well, and KMeans on standardised features keeps the result interpretable. UMAP/HDBSCAN code is left at the bottom in case the dataset is later extended to several metropolises.

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from ftth_equity import config, io
tbl = io.load_parquet(config.PROCESSED / 'buildings_features.parquet')
tbl.shape

## Build the commune-level profile table

One row per commune. Features are structural (size, density, building mix) plus the observed lag rate as the *outcome variable* ,  used to profile clusters, not to feed them.

In [ ]:
com = (
    tbl.groupby('code_insee')
       .agg(
           nom_com=('nom_com', 'first'),
           n_buildings=('imb_id', 'count'),
           n_pm=('pm_ref', 'nunique'),
           buildings_per_pm=('com_buildings_per_pm', 'first'),
           share_individuel=('com_share_individuel', 'first'),
           hhi=('com_hhi', 'first'),
           lag_share=('is_lagging', 'mean'),
       )
       .dropna()
)
com.head()

## Cluster on structural features only

`lag_share` is held out ,  clusters are formed on infrastructure shape, then we check the lag-share distribution per cluster. If the lag concentrates in one cluster, we can describe what kind of commune is at risk without using the target as input.

In [ ]:
feat_cols = ['n_buildings', 'n_pm', 'buildings_per_pm', 'share_individuel', 'hhi']
X = StandardScaler().fit_transform(com[feat_cols])

inertias = []
for k in range(2, 8):
    inertias.append(KMeans(n_clusters=k, n_init=10, random_state=13).fit(X).inertia_)
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(range(2, 8), inertias, marker='o')
ax.set_xlabel('k'); ax.set_ylabel('inertia'); ax.set_title('Elbow plot')
fig.tight_layout()

In [ ]:
K = 4
km = KMeans(n_clusters=K, n_init=10, random_state=13).fit(X)
com['cluster'] = km.labels_
com['cluster'].value_counts().sort_index()

## Cluster profiles

Mean of each feature per cluster, plus the lag share. The clusters that pop out as 'small rural commune' vs 'urban core' should look very different on `n_buildings` and `buildings_per_pm`.

In [ ]:
profile = com.groupby('cluster')[feat_cols + ['lag_share']].mean().round(3)
profile['n_communes'] = com['cluster'].value_counts().sort_index().values
profile

## PCA scatter coloured by cluster, sized by lag

Two dimensions of the standardised feature space, with each commune sized by its observed lag rate. If a single cluster dominates the lag, that cluster's points should be both colour-grouped and visibly larger.

In [ ]:
emb = PCA(n_components=2, random_state=13).fit_transform(X)
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    emb[:, 0], emb[:, 1],
    c=com['cluster'], cmap='tab10',
    s=20 + com['lag_share'].values * 800,
    alpha=0.8, edgecolor='white', linewidth=0.5,
)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title('Communes ,  PCA + KMeans cluster (size = lag rate)')
legend = ax.legend(*scatter.legend_elements(), title='cluster', loc='best')
ax.add_artist(legend)
fig.tight_layout()
fig.savefig(config.FIGURES / 'commune_clusters.png', dpi=140)

## Save labels for the dashboard

In [ ]:
io.save_parquet(com.reset_index(), config.PROCESSED / 'commune_clusters.parquet')

## UMAP / HDBSCAN ,  left as future work

Worth re-running once the dataset spans multiple metropolises. With 80 points HDBSCAN tends to either label everything noise or collapse to one cluster.

In [ ]:
# import umap, hdbscan
# emb = umap.UMAP(n_components=2, random_state=13).fit_transform(X)
# labels = hdbscan.HDBSCAN(min_cluster_size=5).fit_predict(X)
# com.assign(cluster=labels).groupby('cluster').mean()